# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/palkinsuneja/palkin-flyrank-ml-internship-july-to-sept-2026/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip install -q duckdb huggingface_hub pandas numpy

import os
import duckdb
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{hf_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

print("Setup successful.")

Setup successful.


In [4]:
import pandas as pd
import numpy as np

feb = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS gsc_impressions,
    AVG(gsc_avg_position) AS gsc_avg_position,
    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE 0
    END AS ctr
FROM {FEB}
GROUP BY client_hash_id, content_hash_id
""").df()

print("Rows:", len(feb))
display(feb.head())

Rows: 321546


,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr
0,client_e547b89c05043229,content_1eea820697c3b95a,10.678571,12.946228,0.000000
1,client_e547b89c05043229,content_9abd8b303f805847,26.178571,6.495085,0.008186
2,client_e547b89c05043229,content_5f58c55cbfee172a,18.357143,10.490023,0.000000
3,client_e547b89c05043229,content_6fe390ba3af1e456,104.678571,38.436254,0.001024
4,client_e547b89c05043229,content_3ad5d2160242b9ca,34.642857,9.710810,0.002062


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

## 1. Distributions

The key signals have heavy tails, especially impressions. Most content items have relatively low visibility, while a smaller number receive much higher impressions. Average position and CTR are more concentrated but still show meaningful variation. Because of these distributions, simple threshold-based flags should be interpreted as prioritization signals rather than absolute judgments.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signals = ["gsc_impressions", "gsc_avg_position", "ctr"]

summary = feb[signals].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T

display(summary)

print("\nMissing values:")
display(feb[signals].isna().sum())

print("\nKey percentiles:")
for col in signals:
    print(f"\n{col}")
    print(f"Median: {feb[col].median():.4f}")
    print(f"90th percentile: {feb[col].quantile(0.90):.4f}")
    print(f"99th percentile: {feb[col].quantile(0.99):.4f}")

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
gsc_impressions,321546.0,21.955622,107.389437,0.0,0.000000,0.000000,4.428571,43.357143,105.642857,391.412500,7264.321429
gsc_avg_position,153559.0,12.702216,13.977253,0.0,4.731159,7.935419,14.806570,29.192854,42.201347,68.653892,633.000000
ctr,321546.0,0.002370,0.028017,0.0,0.000000,0.000000,0.000000,0.002770,0.005814,0.027027,1.000000



Missing values:


,0
gsc_impressions,0
gsc_avg_position,167987
ctr,0



Key percentiles:

gsc_impressions
Median: 0.0000
90th percentile: 43.3571
99th percentile: 391.4125

gsc_avg_position
Median: 7.9354
90th percentile: 29.1929
99th percentile: 68.6539

ctr
Median: 0.0000
90th percentile: 0.0028
99th percentile: 0.0270


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

## 2. Signal test #1 / #2 / #3

### Signal 1 — High impressions indicate meaningful visibility
Verdict: CONFIRMED

Pages with high impressions have meaningful search visibility, so they are reasonable candidates for closer review when performance is weak.

### Signal 2 — Poor average position identifies an opportunity
Verdict: CONFIRMED

Pages with worse average positions represent a potential discoverability opportunity, especially when they already have meaningful impressions.

### Signal 3 — Low CTR indicates weak search performance
Verdict: MIXED

CTR is useful for identifying potentially weak performance, but CTR varies across pages and should not be treated as a standalone reason to refresh content.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1: high visibility
high_imp = feb[feb["gsc_impressions"] >= feb["gsc_impressions"].quantile(0.75)]

# Signal 2: poor position
poor_pos = feb[feb["gsc_avg_position"] > 10]

# Signal 3: low CTR
low_ctr = feb[feb["ctr"] < 0.02]

print("SIGNAL 1 — High impressions")
print("Rows:", len(high_imp))
print("Median impressions:", round(high_imp["gsc_impressions"].median(), 2))

print("\nSIGNAL 2 — Poor average position")
print("Rows:", len(poor_pos))
print("Median position:", round(poor_pos["gsc_avg_position"].median(), 2))

print("\nSIGNAL 3 — Low CTR")
print("Rows:", len(low_ctr))
print("Median CTR:", round(low_ctr["ctr"].median(), 4))

print("\nSignal test completed.")

SIGNAL 1 — High impressions
Rows: 80460
Median impressions: 28.93

SIGNAL 2 — Poor average position
Rows: 58418
Median position: 18.68

SIGNAL 3 — Low CTR
Rows: 317387
Median CTR: 0.0

Signal test completed.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## 3. The flag-linked test

The flag-linked assumption tested here is that pages with meaningful search visibility but weak average position are useful refresh candidates. The data supports this assumption directionally: high-impression pages with positions worse than 10 have a clear visibility opportunity. However, this is an observational signal and does not prove that refreshing the content will cause rankings or traffic to improve.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Test the flag-linked assumption:
# meaningful visibility + weak average position

high_visibility = feb["gsc_impressions"] >= 1000
weak_position = feb["gsc_avg_position"] > 10

flagged = feb[high_visibility & weak_position].copy()

comparison = pd.DataFrame({
    "group": ["All content", "High visibility + weak position"],
    "rows": [
        len(feb),
        len(flagged)
    ],
    "median_impressions": [
        feb["gsc_impressions"].median(),
        flagged["gsc_impressions"].median()
    ],
    "median_position": [
        feb["gsc_avg_position"].median(),
        flagged["gsc_avg_position"].median()
    ],
    "median_ctr": [
        feb["ctr"].median(),
        flagged["ctr"].median()
    ]
})

display(comparison)

print("Flag-linked test completed.")

,group,rows,median_impressions,median_position,median_ctr
0,All content,321546,0.000000,7.935419,0.0000
1,High visibility + weak position,82,1213.732143,24.196212,0.0017


Flag-linked test completed.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## 4. What this means in practice

Content teams should prioritize pages that already have meaningful search visibility but show weak average position or CTR. These signals are useful for ranking review work, but they should be treated as decision-support indicators rather than proof that a refresh will improve performance. Editors should review the highest-priority pages before taking action.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simple practical priority view

priority = feb[
    (feb["gsc_impressions"] >= 1000) &
    (feb["gsc_avg_position"] > 10)
].copy()

priority["priority_reason"] = np.where(
    priority["ctr"] < 0.02,
    "HIGH_VISIBILITY_WEAK_PERFORMANCE",
    "HIGH_VISIBILITY_POOR_POSITION"
)

priority = priority.sort_values(
    ["gsc_impressions", "gsc_avg_position"],
    ascending=[False, False]
)

print("Priority candidates:", len(priority))
display(priority.head(20))

print("\nNo label-derived or future-window fields are used in this signal audit.")

Priority candidates: 82


,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,priority_reason
179643,client_23a62021009f63c4,content_e8a52cf3d5988c07,5790.321429,13.725133,0.003867,HIGH_VISIBILITY_WEAK_PERFORMANCE
177707,client_23a62021009f63c4,content_36e53e9c707674fc,3597.714286,34.040988,0.002353,HIGH_VISIBILITY_WEAK_PERFORMANCE
179320,client_23a62021009f63c4,content_df47d1b976106de4,3069.107143,18.008694,0.001199,HIGH_VISIBILITY_WEAK_PERFORMANCE
12943,client_fef1a8f436438636,content_84a6bf3578312e90,2856.642857,20.050958,0.000925,HIGH_VISIBILITY_WEAK_PERFORMANCE
179749,client_23a62021009f63c4,content_5e1c049f62e33b11,2609.714286,16.643724,0.001615,HIGH_VISIBILITY_WEAK_PERFORMANCE
14271,client_fef1a8f436438636,content_ba462518dad435fc,2469.071429,27.753576,0.000564,HIGH_VISIBILITY_WEAK_PERFORMANCE
177248,client_23a62021009f63c4,content_3df3f32f3fd58dea,2436.285714,24.899933,0.002272,HIGH_VISIBILITY_WEAK_PERFORMANCE
177986,client_23a62021009f63c4,content_b51957d7f4abe47e,2055.642857,27.816678,0.000625,HIGH_VISIBILITY_WEAK_PERFORMANCE
177180,client_23a62021009f63c4,content_bdf60c86117079be,1833.785714,33.418919,0.000156,HIGH_VISIBILITY_WEAK_PERFORMANCE
14912,client_fef1a8f436438636,content_0aaa197051f58d6f,1782.250000,34.694009,0.000701,HIGH_VISIBILITY_WEAK_PERFORMANCE



No label-derived or future-window fields are used in this signal audit.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.